# NLP Today Session 01: PoS Tagging and Constituency Parsing

### Students:
- Zakaria SOUALAH MOHAMMED
- Wissam AMAR

This notebook studies two related NLP tasks: PoS tagging and constituency parsing. We first compare manual tags with general-purpose and scientific-domain models, then compare a grammar-based CYK parser with Berkeley's learned constituency parser.

In [19]:
import spacy
import benepar as _benepar
import en_core_sci_sm
from typing import List
from pathlib import Path
from shutil import copytree
from tempfile import TemporaryDirectory

# 1. PoS Tagging

## 1.1. Manual Annotation

Below is the manual annotation for the four general-domain sentences:

| Sentence | Manual annotation |
| --- | --- |
| 1 | The/DET cat/NOUN sat/VERB on/ADP the/DET couch/NOUN ./PUNCT |
| 2 | Time/NOUN flies/VERB like/ADP an/DET arrow/NOUN ./PUNCT |
| 3 | The/DET spy/NOUN saw/VERB the/DET cop/NOUN with/ADP the/DET telescope/NOUN ./PUNCT |
| 4 | The/DET spy/NOUN saw/VERB the/DET cop/NOUN with/ADP the/DET revolver/NOUN ./PUNCT |

## 1.2. spaCy Annotation

spaCy's general English model is applied to all six sentences here:

In [14]:
def display_pos_results(title, text, doc):
    """Display one sentence with the same token-level layout in both PoS sections."""
    print(f"\n{title}")
    print(f"Sentence: {text}")
    print(f"{'Token':22} {'Lemma':22} {'PoS':10} {'Dependency':16}")
    print("-" * 74)
    for token in doc:
        print(f"{token.text:22} {token.lemma_:22} {token.pos_:10} {token.dep_:16}")


nlp = spacy.load("en_core_web_sm")
print(f"Loaded model '{nlp.meta['name']}' with pipeline components: {nlp.pipe_names}")

sentences: List[str] = [
    "The cat sat on the couch.",
    "Time flies like an arrow.",
    "The spy saw the cop with the telescope.",
    "The spy saw the cop with the revolver.",
]

for i, sentence in enumerate(sentences, start=1):
    display_pos_results(f"General sentence {i}", sentence, nlp(sentence))

spec_sentences: List[str] = [
    "Arabidopsis thaliana seedlings exhibit longer hypocotyls when they are grown under high ambient temperature, which is defined as thermomorphogenesis.",
    "A spectrogram of PSN J10354824+3900279 obtained on Dec. 19.33 UT suggests that this is a type-Ia at redshift z 0.044.",
]

for i, sentence in enumerate(spec_sentences, start=1):
    display_pos_results(f"Specific sentence {i}", sentence, nlp(sentence))


Loaded model 'core_web_sm' with pipeline components: ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']

General sentence 1
Sentence: The cat sat on the couch.
Token                  Lemma                  PoS        Dependency      
--------------------------------------------------------------------------
The                    the                    DET        det             
cat                    cat                    NOUN       nsubj           
sat                    sit                    VERB       ROOT            
on                     on                     ADP        prep            
the                    the                    DET        det             
couch                  couch                  NOUN       pobj            
.                      .                      PUNCT      punct           

General sentence 2
Sentence: Time flies like an arrow.
Token                  Lemma                  PoS        Dependency      
----------------------

## 1.3. sciSpaCy Annotation

sciSpaCy is designed for scientific and biomedical text. We use it for the two domain-specific sentences and keep the same token table as spaCy so the annotations can be compared directly:

In [17]:
installed_model_path = Path(en_core_sci_sm.__file__).parent / "en_core_sci_sm-0.5.4"
model_temp_dir = TemporaryDirectory()
model_path = Path(model_temp_dir.name) / installed_model_path.name
copytree(installed_model_path, model_path)
config_path = model_path / "config.cfg"
config = config_path.read_text().replace(
    'include_static_vectors = "False"',
    "include_static_vectors = false",
)
config_path.write_text(config)
nlp = spacy.load(model_path)

for index, sentence in enumerate(spec_sentences, start=1):
    doc = nlp(sentence)
    display_pos_results(f"Specific sentence {index}", sentence, doc)
    print("Entities:")
    if doc.ents:
        for entity in doc.ents:
            print(f"- {entity.text} [{entity.label_}]")
    else:
        print("- None detected")



Specific sentence 1
Sentence: Arabidopsis thaliana seedlings exhibit longer hypocotyls when they are grown under high ambient temperature, which is defined as thermomorphogenesis.
Token                  Lemma                  PoS        Dependency      
--------------------------------------------------------------------------
Arabidopsis            arabidopsis            NOUN       compound        
thaliana               thaliana               NOUN       compound        
seedlings              seedling               NOUN       nsubj           
exhibit                exhibit                VERB       ROOT            
longer                 long                   ADV        advmod          
hypocotyls             hypocotyls             ADJ        dobj            
when                   when                   SCONJ      advmod          
they                   they                   PRON       nsubjpass       
are                    be                     AUX        auxpass         
grow

## 2.1. CYK Parsing

The CYK parser uses the manually specified grammar in Chomsky Normal Form and the PoS sequences above. Because the grammar permits both noun-phrase and verb-phrase attachment of a prepositional phrase, the two spy sentences should produce two valid parses each.

In [15]:
class Node():
    def __init__(self, type, left, right=None, word=None):
        self.type = type
        self.left = left
        self.right = right
        self.word = word

    def _recursive_repr(self, tab=0):
        s = f"{'----' * tab}{self.type}"
        if self.word is not None:
            s += f" - {self.word}"
        s += "\n"
        if self.left is not None:
            s += self.left._recursive_repr(tab + 1)
        if self.right is not None:
            s += self.right._recursive_repr(tab + 1)
        return s

    def __str__(self):
        return self._recursive_repr() + "\n"

    def __repr__(self):
        return self._recursive_repr() + "\n"


class CYK():
    def __init__(self, grammar):
        self.grammar = grammar

    def __call__(self, input_list, words):
        size = len(input_list)
        cyk_tab = [[[] for _ in range(i + 1)] for i in range(size)][::-1]

        for i, tags_list in enumerate(input_list):
            for tag in tags_list:
                for production, rule in self.grammar:
                    if len(rule) == 1 and tag in rule:
                        cyk_tab[0][i].append(Node(production, None, word=words[i]))

        for cyk_depth in range(2, size + 1):
            left_levels = list(range(1, cyk_depth))
            right_levels = list(range(1, cyk_depth))[::-1]
            for start in range(size - cyk_depth + 1):
                for left_level, right_level in zip(left_levels, right_levels):
                    left_start = start
                    right_start = start + left_level
                    left_nodes = cyk_tab[left_level - 1][left_start]
                    right_nodes = cyk_tab[right_level - 1][right_start]
                    for production, rule in self.grammar:
                        if len(rule) == 2:
                            left_type, right_type = rule
                            matching_left = [node for node in left_nodes if node.type == left_type]
                            matching_right = [node for node in right_nodes if node.type == right_type]
                            cyk_tab[cyk_depth - 1][start] += [
                                Node(production, left, right)
                                for left in matching_left
                                for right in matching_right
                            ]

        return cyk_tab[-1][0]


# Grammar in Chomsky Normal Form. The NP/VP rules allow both PP attachments.
G = [
    ("ROOT", ("S", "PUNCT")),
    ("S", ("NP", "VP")),
    ("NP", ("DET", "NOUN")),
    ("NP", ("NOUN",)),
    ("NP", ("PROPN", "PROPN")),
    ("NP", ("NP", "PP")),
    ("PP", ("ADP", "NP")),
    ("VP", ("VERB", "NP")),
    ("VP", ("VERB", "PP")),
    ("VP", ("VP", "PP")),
    ("NP", ("NOUN", "NOUN")),
    ("NP", ("NOUN", "PP")),
    ("DET", ("DET",)),
    ("NOUN", ("NOUN",)),
    ("PROPN", ("PROPN",)),
    ("VERB", ("VERB",)),
    ("ADP", ("ADP",)),
    ("PUNCT", ("PUNCT",)),
]

sentences = {
    "The cat sat on the couch.": [
        ["DET"], ["NOUN"], ["VERB"], ["ADP"], ["DET"], ["NOUN"], ["PUNCT"]
    ],
    "Time flies like an arrow.": [
        ["NOUN"], ["VERB"], ["ADP"], ["DET"], ["NOUN"], ["PUNCT"]
    ],
    "The spy saw the cop with the telescope.": [
        ["DET"], ["NOUN"], ["VERB"], ["DET"], ["NOUN"],
        ["ADP"], ["DET"], ["NOUN"], ["PUNCT"]
    ],
    "The spy saw the cop with the revolver.": [
        ["DET"], ["NOUN"], ["VERB"], ["DET"], ["NOUN"],
        ["ADP"], ["DET"], ["NOUN"], ["PUNCT"]
    ],
}

cyk = CYK(G)
for index, (sentence, tags) in enumerate(sentences.items(), start=1):
    tokens = sentence.replace(".", " .").split()
    trees = cyk(tags, tokens)
    print(f"\nCYK sentence {index}")
    print(f"Sentence: {sentence}")
    print(f"Number of parses: {len(trees)}")
    for tree_index, tree in enumerate(trees, start=1):
        print(f"\nParse {tree_index}")
        print(tree)



CYK sentence 1
Sentence: The cat sat on the couch.
Number of parses: 1

Parse 1
ROOT
----S
--------NP
------------DET - The
------------NOUN - cat
--------VP
------------VERB - sat
------------PP
----------------ADP - on
----------------NP
--------------------DET - the
--------------------NOUN - couch
----PUNCT - .



CYK sentence 2
Sentence: Time flies like an arrow.
Number of parses: 1

Parse 1
ROOT
----S
--------NP - Time
--------VP
------------VERB - flies
------------PP
----------------ADP - like
----------------NP
--------------------DET - an
--------------------NOUN - arrow
----PUNCT - .



CYK sentence 3
Sentence: The spy saw the cop with the telescope.
Number of parses: 2

Parse 1
ROOT
----S
--------NP
------------DET - The
------------NOUN - spy
--------VP
------------VERB - saw
------------NP
----------------NP
--------------------DET - the
--------------------NOUN - cop
----------------PP
--------------------ADP - with
--------------------NP
------------------------DET - t

## 2.2. Berkeley Parsing

Berkeley's neural constituency parser uses a learned model rather than the notebook's hand-written grammar. It returns one selected tree for each sentence, including the ambiguous spy examples, which makes the result easy to compare with CYK's alternatives.

In [16]:
nlp = spacy.load("en_core_web_sm")
nlp.add_pipe("benepar", config={"model": "benepar_en3"})

sentences = [
    "The cat sat on the couch.",
    "Time flies like an arrow.",
    "The spy saw the cop with the telescope.",
    "The spy saw the cop with the revolver.",
]

for index, text in enumerate(sentences, start=1):
    doc = nlp(text)
    print(f"\nBerkeley sentence {index}")
    print(f"Sentence: {text}")
    for sent in doc.sents:
        print("Number of parses: 1")
        print("\nParse 1")
        print(sent._.parse_string)


You're using a T5TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.



Berkeley sentence 1
Sentence: The cat sat on the couch.
Number of parses: 1

Parse 1
(S (NP (DT The) (NN cat)) (VP (VBD sat) (PP (IN on) (NP (DT the) (NN couch)))) (. .))

Berkeley sentence 2
Sentence: Time flies like an arrow.
Number of parses: 1

Parse 1
(S (NP (NN Time)) (VP (VBZ flies) (PP (IN like) (NP (DT an) (NN arrow)))) (. .))

Berkeley sentence 3
Sentence: The spy saw the cop with the telescope.
Number of parses: 1

Parse 1
(S (NP (DT The) (NN spy)) (VP (VBD saw) (NP (DT the) (NN cop)) (PP (IN with) (NP (DT the) (NN telescope)))) (. .))

Berkeley sentence 4
Sentence: The spy saw the cop with the revolver.
Number of parses: 1

Parse 1
(S (NP (DT The) (NN spy)) (VP (VBD saw) (NP (DT the) (NN cop)) (PP (IN with) (NP (DT the) (NN revolver)))) (. .))
